# Estudio 02 — IA Simbólica: Compuertas Lógicas y Álgebra de Boole
**Módulo 5: Embedded Systems & TinyML**

---

## ¿Qué es la IA Simbólica?

La Inteligencia Artificial Simbólica (también llamada IA clásica o Good Old-Fashioned AI — GOFAI) se basa en la manipulación de símbolos y reglas lógicas para representar conocimiento y razonar.

### Conceptos fundamentales:
- **Símbolos**: Representan objetos, conceptos o relaciones del mundo real
- **Reglas**: Operaciones lógicas sobre símbolos (SI-entonces)
- **Tablas de verdad**: Describen la salida de una función lógica para todas las combinaciones de entrada

### Álgebra de Boole:
- Operadores básicos: AND (·), OR (+), NOT (¬)
- Operadores derivados: NAND, NOR, XOR, XNOR
- Cualquier función booleana puede implementarse solo con compuertas NAND (universalidad)

### Tabla de verdad genérica:
```
  A   B | AND  OR   NAND  NOR  XOR
  ------+--------------------------
  0   0 |  0    0    1    1    0
  0   1 |  0    1    1    0    1
  1   0 |  0    1    1    0    1
  1   1 |  1    1    0    0    0
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from libreria_modulo5 import *
import numpy as np

In [ ]:
configurar_reproducibilidad(42)

## Tablas de verdad de compuertas básicas

In [ ]:
tabla_verdad(compuerta_and, 'AND')
tabla_verdad(compuerta_or, 'OR')
tabla_verdad(compuerta_not, 'NOT', 1)

In [ ]:
tabla_verdad(compuerta_xor, 'XOR')
tabla_verdad(compuerta_nand, 'NAND')

## Universalidad de la compuerta NAND

La compuerta NAND es **universal**: cualquier función booleana puede construirse solo con NANDs.

### NOT desde NAND:
  `NOT(A) = NAND(A, A)`

In [ ]:
def NOT_desde_NAND(a):
    return compuerta_nand(a, a)

print("NOT construido con NAND:")
print("  A | NOT(A)")
print("  ---+-------")
for a in [0, 1]:
    print(f"  {a} |    {NOT_desde_NAND(a)}")

### AND desde NAND:
  `AND(A, B) = NOT(NAND(A, B)) = NAND(NAND(A, B), NAND(A, B))`

In [ ]:
def AND_desde_NAND(a, b):
    return compuerta_not(compuerta_nand(a, b))

print("AND construido con NAND:")
print("  A   B | AND")
print("  ------+-----")
for a in [0, 1]:
    for b in [0, 1]:
        print(f"  {a}   {b} |   {AND_desde_NAND(a, b)}")

### OR desde NAND:
  `OR(A, B) = NAND(NOT(A), NOT(B)) = NAND(NAND(A,A), NAND(B,B))`

In [ ]:
def OR_desde_NAND(a, b):
    return compuerta_nand(compuerta_not(a), compuerta_not(b))

print("OR construido con NAND:")
print("  A   B | OR")
print("  ------+----")
for a in [0, 1]:
    for b in [0, 1]:
        print(f"  {a}   {b} |  {OR_desde_NAND(a, b)}")

## Circuitos aritméticos con compuertas lógicas

### Sumador Medio (Half Adder)
Suma 2 bits → produce **suma** (XOR) y **acarreo** (AND):
```
  S = A ⊕ B
  C = A · B
```

In [ ]:
print("=== Sumador Medio ===")
print("  A   B | Suma  Acarreo")
print("  ------+--------------")
for a in [0, 1]:
    for b in [0, 1]:
        r = circuito_sumador_medio(a, b)
        print(f"  {a}   {b} |   {r['suma']}      {r['acarreo']}")

### Sumador Completo (Full Adder)
Suma 3 bits (A + B + acarreo_entrada) → suma y acarreo de salida.
Es la base de los sumadores en los CPUs.

In [ ]:
print("=== Sumador Completo ===")
print("  A   B  Cin | Suma  Cout")
print("  ----------+------------")
for a in [0, 1]:
    for b in [0, 1]:
        for c in [0, 1]:
            r = circuito_sumador_completo(a, b, c)
            print(f"  {a}   {b}    {c} |   {r['suma']}      {r['acarreo']}")

## Leyes de De Morgan

Permiten convertir entre expresiones AND/OR usando negaciones:

1. **¬(A · B) = ¬A + ¬B**  (NAND = NOT A OR NOT B)
2. **¬(A + B) = ¬A · ¬B**  (NOR = NOT A AND NOT B)

### Demostración con código:

In [ ]:
print("=== Leyes de De Morgan ===")
print()
print("1. ¬(A · B) = ¬A + ¬B")
print("  A   B | ¬(A·B)  ¬A+¬B")
print("  ------+--------------")
for a in [0, 1]:
    for b in [0, 1]:
        izq = compuerta_nand(a, b)
        der = compuerta_or(compuerta_not(a), compuerta_not(b))
        print(f"  {a}   {b} |   {izq}      {der}  {'✓' if izq == der else '✗'}")

print()
print("2. ¬(A + B) = ¬A · ¬B")
print("  A   B | ¬(A+B)  ¬A·¬B")
print("  ------+--------------")
for a in [0, 1]:
    for b in [0, 1]:
        izq = compuerta_nor(a, b)
        der = compuerta_and(compuerta_not(a), compuerta_not(b))
        print(f"  {a}   {b} |   {izq}      {der}  {'✓' if izq == der else '✗'}")

## Leyes del Álgebra de Boole

### Leyes básicas:

| Ley | AND | OR |
|-----|-----|----|
| Identidad | A · 1 = A | A + 0 = A |
| Elemento nulo | A · 0 = 0 | A + 1 = 1 |
| Idempotencia | A · A = A | A + A = A |
| Complemento | A · ¬A = 0 | A + ¬A = 1 |
| Involución | ¬(¬A) = A | — |

### Propiedades:
- **Conmutativa**: A · B = B · A, A + B = B + A
- **Asociativa**: (A·B)·C = A·(B·C), (A+B)+C = A+(B+C)
- **Distributiva**: A·(B+C) = (A·B)+(A·C), A+(B·C) = (A+B)·(A+C)
- **Absorción**: A·(A+B) = A, A+(A·B) = A

### Simplificación booleana:

Ejemplo: `S = A·B + A·¬B`

```
  S = A·B + A·¬B
  S = A·(B + ¬B)       (Distributiva)
  S = A·1              (Complemento)
  S = A                (Identidad)
```

Esto significa que la salida solo depende de A, no de B.

In [ ]:
print("=== Simplificación: A·B + A·¬B ===")
print("  A   B | A·B  A·¬B  S  Simplificado (A)")
print("  ------+-------------------------------")
for a in [0, 1]:
    for b in [0, 1]:
        term1 = compuerta_and(a, b)
        term2 = compuerta_and(a, compuerta_not(b))
        s = compuerta_or(term1, term2)
        print(f"  {a}   {b} |  {term1}    {term2}   {s}        {a}           {'✓' if s == a else '✗'}")

## Ejercicio #TODO — Sumador de 2 bits

**Objetivo**: Construir un sumador de 2 bits (A₁A₀ + B₁B₀) usando compuertas lógicas.

### Especificaciones:
- Entradas: A₁, A₀, B₁, B₀ (cada uno 0/1)
- Salidas: S₂, S₁, S₀ (3 bits, rango 0–6)

### Diagrama:
```
  A₁  B₁        A₀  B₀
   │   │         │   │
   ▼   ▼         ▼   ▼
 ┌────────┐    ┌────────┐
 │ Full   │ ◄──│ Half   │
 │ Adder  │    │ Adder  │
 └───┬────┘    └───┬────┘
     │ Cout   Sum  │ Sum
     ▼             ▼
    S₂            S₁     S₀
```

### Pistas:
```python
def sumador_2bits(A1, A0, B1, B0):
    # Bit 0: medio sumador para A0 + B0
    r0 = circuito_sumador_medio(A0, B0)
    S0 = r0['suma']
    acarreo = r0['acarreo']
    
    # Bit 1: sumador completo para A1 + B1 + acarreo
    r1 = circuito_sumador_completo(A1, B1, acarreo)
    S1 = r1['suma']
    S2 = r1['acarreo']  # Acarreo final
    
    return {'S2': S2, 'S1': S1, 'S0': S0}
```

> ⚠️ **Reto extra**: Generaliza a un sumador de 4 bits o implementa un restador.